### Cell 1: 初始化与配置

In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

# 引入自定义模块
from kg_pipeline import KGPipeline
from df_helpers import ResolveCoreferences, ExtractConcepts, df2Graph, graph2Df, contextual_proximity

# --- 配置 ---
# 请根据实际路径修改 INPUT_FILE
INPUT_FILE = "../data_input/dataset/musique/musique_1000_test.json"
OUTPUT_DIR = "../data_output/dataset/musique/ds1000"

# 设置处理数量上限 (None 表示处理全部)
MAX_CONTEXTS = 1000
# 是否强制重新生成 (True: 忽略缓存重新运行; False: 优先加载缓存)
REGENERATE = False  

# 初始化 Pipeline 实例
pipeline = KGPipeline(output_dir=OUTPUT_DIR)

d:\Anaconda_envs\envs\andelie\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Step 1 - 数据加载与分块 (Chunking)

In [2]:
# 将原始 JSON 数据加载并分割为 Chunks
if REGENERATE:
    df_chunks = pipeline.load_and_split_data(
        json_path=INPUT_FILE, 
        max_contexts=MAX_CONTEXTS
    )
else:
    # 尝试加载缓存
    chunk_cache = Path(OUTPUT_DIR) / "chunk.csv"
    if chunk_cache.exists():
        df_chunks = pd.read_csv(chunk_cache, sep="|")
    else:
        # 如果缓存不存在，强制重新生成
        df_chunks = pipeline.load_and_split_data(json_path=INPUT_FILE, max_contexts=MAX_CONTEXTS)

print(f"Loaded {len(df_chunks)} chunks.")
print(df_chunks.head(2))

Loaded 19998 chunks.
   context_id  chunk_id                           title  \
0           0         0             Grant's First Stand   
1           0         1  List of show business families   

                                                text  
0  Grant's First Stand information: Grant's First...  
1  List of show business families information: Ac...  


## Step 2 - 指代消解 (Coreference Resolution)

In [3]:
# 对 Chunks 进行指代消解
resolved_path = Path(OUTPUT_DIR) / "resolved_chunks.csv"

if REGENERATE or not resolved_path.exists():
    df_chunks = ResolveCoreferences(df_chunks,max_workers=30)
    df_chunks.to_csv(resolved_path, sep="|", index=False)
else:
    print("Loading resolved chunks from cache...")
    df_chunks = pd.read_csv(resolved_path, sep="|")

print(df_chunks[['text', 'resolved_text']].head(2))

Loading resolved chunks from cache...
                                                text  \
0  Grant's First Stand information: Grant's First...   
1  List of show business families information: Ac...   

                                       resolved_text  
0  Grant's First Stand information: Grant's First...  
1  List of show business families information: Ac...  


## Step 3 - 生成 Chunk 嵌入向量 (Embeddings)

In [ ]:
# REGENERATE = True
# 定义文件名
final_emb_filename = "chunks_with_embeddings.parquet"
temp_title_filename = "temp_title_embeddings.parquet" # 临时文件
final_path = Path(OUTPUT_DIR) / final_emb_filename
temp_path = Path(OUTPUT_DIR) / temp_title_filename

# 1. 只有 REGENERATE 为 True 时才重新计算，否则直接加载
if REGENERATE or not final_path.exists():
    print("🚀 [Notebook] 开始双重向量计算 (Content + Title)...")

    # --- 第一步：计算正文 (Content) 向量 ---
    # 这会生成 'embedding' 列
    df_content = pipeline.compute_embeddings(
        df=df_chunks, 
        text_col='text', 
        id_col='chunk_id', 
        file_name=final_emb_filename 
    )

    # --- 第二步：计算标题 (Title) 向量 ---
    # 为了不覆盖原来的文件，我们先存到一个临时文件里
    # 注意：pipeline 默认会把列名存为 'embedding'
    df_title = pipeline.compute_embeddings(
        df=df_chunks, 
        text_col='title', 
        id_col='chunk_id', 
        file_name=temp_title_filename 
    )

    # --- 第三步：在 Notebook 中执行合并与重命名 ---
    print("🔄 [Notebook] 正在合并 Title 向量...")
    
    # 提取 df_title 中的 chunk_id 和 embedding，并将 embedding 改名为 title_embedding
    # 注意：这里我们只取需要的两列，防止其他列重复
    df_title_subset = df_title[['chunk_id', 'embedding']].copy()
    df_title_subset.rename(columns={'embedding': 'title_embedding'}, inplace=True)
    
    # 将标题向量 merge 到主数据中
    df_final = pd.merge(df_content, df_title_subset, on='chunk_id', how='left')
    
    # --- 第四步：手动保存合并后的结果 ---
    # 必须把 numpy array 转回 list 才能存 Parquet (这是 Pipeline 内部做的事，我们在外面手动做一遍)
    for col in ['embedding', 'title_embedding']:
        if col in df_final.columns:
            # 检查第一行如果是 numpy 数组，则执行转换
            if not df_final[col].empty and isinstance(df_final[col].iloc[0], np.ndarray):
                df_final[col] = df_final[col].apply(lambda x: x.tolist())

    df_final.to_parquet(final_path, index=False)
    
    # 清理临时文件
    if temp_path.exists():
        os.remove(temp_path)
        
    print(f"✅ [Notebook] 双重向量合并完成！已保存至 {final_path}")
    df_chunks = df_final # 更新当前内存中的 df_chunks

else:
    print(f"🔍 [Notebook] 加载现有缓存: {final_path}")
    df_chunks = pd.read_parquet(final_path)

# --- 验证数据 ---
print("-" * 30)
first_row = df_chunks.iloc[0]
print(f"Data Loaded. Columns: {df_chunks.columns.tolist()}")
if 'embedding' in df_chunks.columns:
    print(f"Content Vector Dim: {len(first_row['embedding'])}")
if 'title_embedding' in df_chunks.columns:
    print(f"Title Vector Dim:   {len(first_row['title_embedding'])}")

🚀 [Notebook] 开始双重向量计算 (Content + Title)...
🔍 [Pipeline] 加载嵌入缓存: ..\data_output\dataset\musique\ds1000\chunks_with_embeddings.parquet
⏳ [Pipeline] 开始计算 19998 条数据的嵌入向量...


Computing Embeddings:  27%|██▋       | 83/313 [01:24<03:42,  1.04it/s]

## Step 4 - 概念/实体抽取 (Concept Extraction)

In [5]:
REGENERATE = False
# 并行从文本中抽取实体
concepts_path = Path(OUTPUT_DIR) / "extracted_concepts.csv"

if REGENERATE or not concepts_path.exists():
    df_concepts = ExtractConcepts(dataframe=df_chunks, max_workers=30)
    df_concepts.to_csv(concepts_path, sep="|", index=False)
else:
    df_concepts = pd.read_csv(concepts_path, sep="|")

print(f"Extracted {len(df_concepts)} concepts.")

Extracted 224351 concepts.


## Step 5 - 实体标准化与对齐 (Entity Standardization)

In [6]:
# 这一步包含了最复杂的逻辑：
# 1. 聚合实体同义词 (Rich Text)
# 2. 计算实体 Embedding
# 3. 使用 HAC + Genealogy Penalty对齐
# 4. 生成标准实体名称
# 这一步包含了最复杂的逻辑：聚合同义词 -> 计算Embedding -> HAC + Genealogy Penalty -> 生成标准名称
REGENERATE = False

std_path = Path(OUTPUT_DIR) / "dp_extracted_concepts.csv"
if REGENERATE or not std_path.exists():
    # 执行标准化流程 (内部会保存文件)
    df_concepts_std = pipeline.standardize_entities(df_concepts)
else:
    print(f"🔍 加载缓存的标准化实体: {std_path}")
    df_concepts_std = pd.read_csv(std_path, sep="|")

print("Standardization sample:")
print(df_concepts_std[['Entity', 'Standard_Entity', 'cluster_id']].head())

🔍 加载缓存的标准化实体: ..\data_output\dataset\musique\ds1000\dp_extracted_concepts.csv
Standardization sample:
          Entity Standard_Entity  cluster_id
0           2001            2001         168
1  First Session   First Session         191
2           1961            1961         178
3      Blue Note       Blue Note          82
4    Grant Green     Grant Green         121


## Step 6 - QA 数据提取

In [7]:
# 从原始数据提取问题和答案对
qa_path = Path(OUTPUT_DIR) / "qa.csv"


if REGENERATE or not qa_path.exists():
    df_qa = pipeline.extract_qa_pairs(INPUT_FILE, max_contexts=MAX_CONTEXTS)
else:
    print(f"🔍 加载缓存的 QA 数据: {qa_path}")
    df_qa = pd.read_csv(qa_path, sep="|")

print(df_qa.head(2))

🔍 加载缓存的 QA 数据: ..\data_output\dataset\musique\ds1000\qa.csv
                                            question            answer  \
0          Who is the spouse of the Green performer?  Miquette Giraudy   
1  Who founded the company that distributed the f...      Mike Medavoy   

   context_id  
0           0  
1           1  


### Step 7 - 基础图谱构建 (Relation Extraction)

In [8]:
# 1. 准备实体映射字典 {chunk: {original_name: standard_name}}
entity_map = pipeline.generate_entity_map_for_graph(df_concepts_std)

# REGENERATE = True

# 2. 执行关系抽取
graph_path = Path(OUTPUT_DIR) / "graph.csv"

if REGENERATE or not graph_path.exists():
    # 使用 LLM 抽取关系，传入 entity_map 以实现引导式抽取
    relations_list = df2Graph(df_chunks, entity_map,max_workers=30)
    df_graph = graph2Df(relations_list)
    df_graph.to_csv(graph_path, sep="|", index=False)
else:
    df_graph = pd.read_csv(graph_path, sep="|")

print(f"Extracted {len(df_graph)} relations.")
print(df_graph.head())

d:\Code\jupyter\knowledge_graph\musique\kg_pipeline.py:379: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return valid.groupby(['context_id', 'chunk_id']).apply(


Extracted 190108 relations.
   context_id               node_1       node_2  \
0           0        First Session         2001   
1           0        First Session    Blue Note   
2           0  Grant's First Stand         1961   
3           0  Grant's First Stand  Grant Green   
4           0  Grant's First Stand    Blue Note   

                                                edge  chunk_id  
0                First Session was released in 2001.         0  
1  Earlier recordings made by Grant for Blue Note...         0  
2          Grant's First Stand was released in 1961.         0  
3  Grant's First Stand is the debut album by Amer...         0  
4  Grant's First Stand was recorded and released ...         0  


### Step 8 - 计算上下文共现关系 (Contextual Proximity)

In [9]:
# 基于 Chunk 共现计算额外的图谱边
# REGENERATE = True

df_proximity = contextual_proximity(df_graph)
df_proximity.to_csv(Path(OUTPUT_DIR) / "contextual_proximity.csv", sep="|", index=False)

print(f"Generated {len(df_proximity)} proximity edges.")
print(df_proximity.head())

Generated 392716 proximity edges.
   context_id                                          node_1  \
0         641  "(Can't Live Without Your) Love and Affection"   
2         641  "(Can't Live Without Your) Love and Affection"   
4         641  "(Can't Live Without Your) Love and Affection"   
5         641  "(Can't Live Without Your) Love and Affection"   
16        973                              "10,000 to 15,000"   

                   node_2 chunk_id  count                  edge  
0        "After the Rain"    12836      8  contextual_proximity  
2           Gunnar Nelson    12836     10  contextual_proximity  
4          Matthew Nelson    12836     10  contextual_proximity  
5                  Nelson    12836     10  contextual_proximity  
16  Battle of Blood River    19476      7  contextual_proximity  


## Step 9 - Merge Entities & Compute Dual Embeddings

In [ ]:
# --- New Step: Merge and Enrich ---
# 聚合实体信息并计算向量 (vec_entity 和 vec_desc)
# REGENERATE = True
merged_path = Path(OUTPUT_DIR) / "concepts_merged_with_vectors.parquet"

# 1. 如果强制重新生成，先删除缓存文件
if REGENERATE and merged_path.exists():
    print(f"🗑️ [REGENERATE] 删除旧聚合缓存: {merged_path}")
    os.remove(merged_path)

# 2. 调用 Pipeline (内部逻辑: 无文件则计算并保存，有文件则加载)
df_merged_vectors = pipeline.merge_and_embed_concepts(df_concepts=df_concepts_std)

print("Merged Data Sample:")
print(df_merged_vectors[['Standard_Entity', 'description', 'vec_entity']].head(2))

🚀 [Pipeline] 开始实体聚合...
🔄 [Helper] 正在聚合实体数据...
✅ [Helper] 聚合完成。原始行数: 224351 -> 聚合后行数: 192709
🛠️ [Pipeline] 构建增强实体文本...
🚀 [Pipeline] 计算 Enriched Entity Vectors...


Vec: Entity: 100%|██████████| 3012/3012 [59:15<00:00,  1.18s/it]  


🚀 [Pipeline] 计算 Description Vectors...


Vec: Desc: 100%|██████████| 3012/3012 [52:05<00:00,  1.04s/it] 


💾 [Pipeline] 保存聚合向量表到: ..\data_output\dataset\musique\ds1000\concepts_merged_with_vectors.parquet
Merged Data Sample:
  Standard_Entity                                        description  \
0            1961  The year Grant Green's debut album, Grant's Fi...   
1       Blue Note  The record label that released Grant Green's d...   

                                          vec_entity  
0  [0.01651139, 0.025822053, -0.026199149, -0.008...  
1  [-0.036208298, 0.026029117, -0.04892347, 0.013...  
